In [2]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import re

In [12]:
headers = {
    "User-Agent": "Mozilla/5.0"
}
base_url = "https://www.thegoodguys.com.au/fridges-and-freezers/refrigerators?page="
data = []

for page in range(1,7):
    url = base_url + str(page)
    print('Scraping page {page} ... ')

    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    products = soup.find_all('div', class_='product-tile-inner')

    if not products:
        print('NO MORE PRODUCTS FOUND! STOPPING...')
        break

    for product in products:
        title = product.find('h3', class_='product-tile-name').get_text(strip=True)
        price_text = product.find('div', class_='product-tile__price-label pricepoint-promolabel-wrap').get_text(strip=True)
        prices = re.findall(r'\$\d+', price_text)
        price= prices[0]
        reviews_num = product.find('span', class_ = 'reviews-stars-link__text-link text-medium marg-left-5')
        full_stars = product.find_all('i', class_ = 'ggdsicon-star')
        half_stars = product.find_all('i', class_ = 'ggdsicon-star-half-o')
        links_tag = product.find('a', class_='disp-block')
        links = links_tag['href']
        full_link =  links
        rating = len(full_stars) + 0.5*len(half_stars)
        

        data.append({'Title':title,'Price':price,'Reviews count': reviews_num,'Rating' : rating,'Link' : full_link
        })
df=pd.DataFrame(data)

Scraping page {page} ... 
Scraping page {page} ... 
Scraping page {page} ... 
Scraping page {page} ... 
Scraping page {page} ... 
Scraping page {page} ... 


In [14]:
df['Reviews count'] = df['Reviews count'].fillna(0).astype(str).str.replace(r'\D','', regex=True).replace('','0').astype(int)
df['Price']=df['Price'].replace(r'\D','',regex=True).astype(int)
pd.set_option('display.max_rows',200)
pd.set_option('display.max_columns', 10)

In [15]:
print(df.head())
print(f'Total Products scraped: {len(df)} ')

                             Title  Price  Reviews count  Rating  \
0  TCL 197L Top Mount Refrigerator    499            516     4.0   
1   LG 243L Top Mount Refrigerator    599            587     4.5   
2  TCL 415L Top Mount Refrigerator    699            520     4.5   
3           Hisense 45L Bar Fridge    199              0     0.0   
4          Hisense 124L Bar Fridge    249              0     0.0   

                                                Link  
0  https://www.thegoodguys.com.au/tcl-197l-top-mo...  
1  https://www.thegoodguys.com.au/lg-243l-top-mou...  
2  https://www.thegoodguys.com.au/tcl-415l-top-mo...  
3  https://www.thegoodguys.com.au/hisense-45l-bar...  
4  https://www.thegoodguys.com.au/hisense-124l-ba...  
Total Products scraped: 180 


In [21]:
Eligible_Products = df[(df['Price']<1200)&(df['Reviews count']>1000)&(df['Rating']>3.5)]


10

In [22]:
Eligible_Products

,Title,Price,Reviews count,Rating,Link
18,LG 420L Bottom Mount Refrigerator,898,5556,4.5,https://www.thegoodguys.com.au/lg-420l-bottom-...
21,Hisense 496L Top Mount Refrigerator,999,5307,4.5,https://www.thegoodguys.com.au/hisense-496l-to...
42,CHiQ 90L Bar Fridge,249,5181,4.5,https://www.thegoodguys.com.au/chiq-90l-bar-fr...
46,CHiQ 118L Top Mount Refrigerator,349,5306,4.0,https://www.thegoodguys.com.au/chiq-118l-top-m...
49,Hisense 179L Bar Fridge,399,5126,4.5,https://www.thegoodguys.com.au/hisense-179l-ba...
54,Hisense 242L All Fridge,699,5168,4.5,https://www.thegoodguys.com.au/hisense-242l-al...
62,LG 315L Top Mount Refrigerator,899,5241,4.5,https://www.thegoodguys.com.au/lg-315l-top-mou...
66,LG 375L Top Mount Refrigerator,999,5242,4.5,https://www.thegoodguys.com.au/lg-375l-top-mou...
73,Westinghouse 425L Bottom Mount Refrigerator,1149,5250,4.5,https://www.thegoodguys.com.au/westinghouse-42...
74,LG 420L Bottom Mount Refrigerator,1199,51149,4.5,https://www.thegoodguys.com.au/lg-420l-bottom-...


In [24]:
Eligible_Products.to_csv(r'C:\Users\Bhupal\OneDrive\Desktop\GoodGuys_Fridges Project\Eligible_Fridges.csv', index=False)